In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv  # 추가된 부분

def fetch_dart_major_reports_final(api_key, start_date, end_date):
    """
    OpenDART API를 통해 특정 기간의 '주요사항보고서(B)'를 수집하고,
    정교화된 키워드를 바탕으로 2가지 이벤트 플래그를 분리하여 태깅합니다.
    """
    url = "https://opendart.fss.or.kr/api/list.json"
    
    params = {
        'crtfc_key': api_key,
        'bgn_de': start_date,   
        'end_de': end_date,     
        'pblntf_ty': 'B',       # 주요사항보고서
        'page_no': 1,
        'page_count': 100       
    }
    
    print(f"[{start_date} ~ {end_date}] DART 주요사항보고서(B) 수집 시작...")
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        if data.get('status') == '000': 
            df = pd.DataFrame(data['list'])
            df = df[['corp_name', 'corp_code', 'stock_code', 'report_nm', 'rcept_dt']]
            df.columns = ['기업명', '고유번호', '종목코드', '보고서명', '접수일자']
            
            # 자본이벤트 및 상장폐지 관련 키워드 리스트
            CAPITAL_EVENT_KEYWORDS = ["유상증자", "무상증자", "전환사채", "신주인수권부사채", "교환사채"]
            DELISTING_KEYWORDS = ["감사의견", "관리종목", "상장폐지", "거래정지", "실질심사"]
            
            # 1. 자본이벤트 태깅
            df['capital_event_flag'] = df['보고서명'].apply(
                lambda title: 1 if any(k in str(title) for k in CAPITAL_EVENT_KEYWORDS) else 0
            )
            
            # 2. 상장폐지/관리종목 관련 태깅
            df['delisting_related_flag'] = df['보고서명'].apply(
                lambda title: 1 if any(k in str(title) for k in DELISTING_KEYWORDS) else 0
            )
            
            print(f"✅ 총 {len(df)}건의 데이터를 성공적으로 수집 및 태깅했습니다.")
            return df
        
        else:
            print(f"❌ API 응답 실패: {data.get('message')}")
            return pd.DataFrame()
    else:
        print(f"❌ 네트워크 연결 에러 (상태 코드: {response.status_code})")
        return pd.DataFrame()

# ==========================================
# 실행 영역 (.env 적용)
# ==========================================
if __name__ == "__main__":
    # 💡 1. 같은 폴더에 있는 .env 파일을 읽어옵니다.
    load_dotenv()
    
    # 💡 2. 환경 변수에서 키 값을 꺼내옵니다. (코드에 키가 노출되지 않음!)
    MY_DART_API_KEY = os.getenv("DART_API_KEY")
    
    # 키가 제대로 불러와졌는지 안전 장치 추가
    if not MY_DART_API_KEY:
        print("🚨 에러: .env 파일에서 'DART_API_KEY'를 찾을 수 없습니다. 파일을 확인해주세요.")
    else:
        df_dart = fetch_dart_major_reports_final(
            api_key=MY_DART_API_KEY, 
            start_date="20231001", 
            end_date="20231031"
        )
        
        if not df_dart.empty:
            display(df_dart.head())
            
            filename = "dart_separated_events.csv"
            df_dart.to_csv(filename, index=False, encoding="utf-8-sig")

ModuleNotFoundError: No module named 'pandas'